# WeaveGuard AI — Étape A : audit des données
Ce notebook lit tes datasets sur Google Drive et produit un **rapport chiffré** (`audit_report.json` + `audit_summary.txt`).
**À la fin, copie-colle ici le contenu de la dernière cellule** (ou uploade `audit_report.json`).

Structure attendue sur Drive (modifiable dans la cellule 2) :
```
MyDrive/WeaveGuard/data/
  BDFD9/  textile_defects/  hole_detection/   (datasets YOLO)
  dtd/    (images/  labels/  imdb/)           (Describable Textures Dataset)
```
Exécute les cellules dans l'ordre. Aucune donnée n'est modifiée (lecture seule).

In [ ]:
# 1) Dépendances + montage de Drive
!pip -q install pyyaml openpyxl
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Configuration — adapte ROOT si ton dossier Drive est différent
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/WeaveGuard/data')
OUT = ROOT.parent / 'audit_output'
OUT.mkdir(exist_ok=True, parents=True)

YOLO_DATASETS = ['BDFD9', 'textile_defects', 'hole_detection']
DTD_DIR = ROOT / 'dtd' if (ROOT / 'dtd').exists() else ROOT   # fallback si images/labels/imdb sont directement dans ROOT
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
NEAR_DUP_MAX_HAMMING = 3        # distance de Hamming max (dhash 64 bits) pour "quasi-doublon"
MAX_IMAGES_NEAR_DUP = 25000     # au-delà, la détection de quasi-doublons est sautée
REPORT = {}

print('ROOT existe :', ROOT.exists())
if ROOT.exists():
    print('Contenu :', sorted(p.name for p in ROOT.iterdir()))

In [ ]:
# 3) Arborescence
import collections
def show_tree(p, max_depth=3, depth=0, max_dirs=12):
    try:
        entries = sorted(p.iterdir(), key=lambda x: x.name.lower())
    except Exception as e:
        print('  ' * depth + '! erreur', e); return
    dirs = [e for e in entries if e.is_dir()]
    files = [e for e in entries if e.is_file()]
    exts = dict(collections.Counter(f.suffix.lower() or '(sans ext)' for f in files))
    print('  ' * depth + f'{p.name}/  [{len(files)} fichiers {exts}]')
    if depth < max_depth:
        for d in dirs[:max_dirs]:
            show_tree(d, max_depth, depth + 1, max_dirs)
        if len(dirs) > max_dirs:
            print('  ' * (depth + 1) + f'... +{len(dirs) - max_dirs} autres dossiers')

for name in YOLO_DATASETS + [DTD_DIR.name]:
    d = ROOT / name if name != DTD_DIR.name else DTD_DIR
    print('=' * 70)
    if d.exists(): show_tree(d)
    else: print(f'{name} : INTROUVABLE ({d})')

In [ ]:
# 4) Fichiers texte clés : data.yaml, classes.txt, README.md
for name in YOLO_DATASETS:
    d = ROOT / name
    if not d.exists(): continue
    for fname in ['data.yaml', 'classes.txt', 'README.md', 'readme.md']:
        f = d / fname
        if f.exists():
            print('=' * 70); print(f'{name}/{fname}')
            print(f.read_text(encoding='utf-8', errors='replace')[:2500])

In [ ]:
# 5) Audit YOLO (version rapide : lecture parallèle + barre de progression + cache)
import yaml, hashlib, io, json, pickle
import numpy as np, pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

N_THREADS = 32          # la lecture Drive est limitée par le réseau : plus de threads = plus rapide
USE_CACHE = True
CACHE = OUT / 'scan_cache.pkl'

def load_names(ds_dir):
    for y in list(ds_dir.glob('*.yaml')) + list(ds_dir.glob('*.yml')):
        try:
            d = yaml.safe_load(open(y, encoding='utf-8'))
            n = d.get('names') if isinstance(d, dict) else None
            if isinstance(n, dict): n = [n[k] for k in sorted(n, key=lambda k: int(k))]
            if n: return list(n)
        except Exception:
            pass
    f = ds_dir / 'classes.txt'
    if f.exists():
        return [l.strip() for l in open(f, encoding='utf-8') if l.strip()]
    return None

def split_of(parts):
    for p in parts:
        q = p.lower()
        if q in ('train', 'training'): return 'train'
        if q in ('val', 'valid', 'validation'): return 'val'
        if q == 'test': return 'test'
    return 'unknown'

def label_path(ds_dir, rel):
    parts = list(rel.parts)
    idx = [i for i, p in enumerate(parts) if p.lower() == 'images']
    if idx: parts[idx[-1]] = 'labels'
    return (ds_dir / Path(*parts)).with_suffix('.txt')

def dhash(im):
    g = np.asarray(im.convert('L').resize((9, 8)), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    v = 0
    for b in bits: v = (v << 1) | int(b)
    return v

def process(args):
    name, d, names, p = args
    rel = p.relative_to(d); split = split_of(rel.parts)
    rec = dict(dataset=name, split=split, path=str(rel), md5=None, dhash=None, w=None, h=None,
               corrupt=False, status=None, n_boxes=0)
    lboxes, lbad = [], []
    try:
        data = p.read_bytes()
        rec['md5'] = hashlib.md5(data).hexdigest()
        with Image.open(io.BytesIO(data)) as im:
            rec['w'], rec['h'] = im.size
            rec['dhash'] = dhash(im)
    except Exception:
        rec['corrupt'] = True
    lp = label_path(d, rel)
    if not lp.exists():
        rec['status'] = 'missing_label'
    else:
        nb = 0
        for ln, line in enumerate(open(lp, encoding='utf-8', errors='replace').read().splitlines(), 1):
            s = line.split()
            if not s: continue
            try:
                c = int(float(s[0])); vals = [float(x) for x in s[1:]]
                if len(vals) == 4:
                    x, y, w, h = vals
                elif len(vals) >= 6 and len(vals) % 2 == 0:      # polygone
                    xs, ys = vals[0::2], vals[1::2]
                    x, y, w, h = (min(xs)+max(xs))/2, (min(ys)+max(ys))/2, max(xs)-min(xs), max(ys)-min(ys)
                else:
                    raise ValueError('nb colonnes')
                problems = []
                if names and not (0 <= c < len(names)): problems.append('classe hors plage')
                if not (0 < w <= 1 and 0 < h <= 1): problems.append('w/h invalide')
                if not (0 <= x <= 1 and 0 <= y <= 1): problems.append('centre hors image')
                if problems:
                    lbad.append(dict(dataset=name, file=str(lp.relative_to(d)), line=ln, problem=', '.join(problems)))
                    continue
                nb += 1
                lboxes.append(dict(dataset=name, split=split, cls=c,
                                   cls_name=(names[c] if names and c < len(names) else str(c)),
                                   w=w, h=h, area=w*h, wpx=w*(rec['w'] or 0), hpx=h*(rec['h'] or 0)))
            except Exception:
                lbad.append(dict(dataset=name, file=str(lp.relative_to(d)), line=ln, problem='ligne illisible'))
        rec['n_boxes'] = nb
        rec['status'] = 'labeled' if nb > 0 else 'empty_label'
    return rec, lboxes, lbad, lp

if USE_CACHE and CACHE.exists():
    records, boxes, bad_lines, orphan_labels, class_names = pickle.load(open(CACHE, 'rb'))
    print('Résultats rechargés depuis le cache :', CACHE)
else:
    records, boxes, bad_lines, orphan_labels, class_names = [], [], [], [], {}
    for name in YOLO_DATASETS:
        d = ROOT / name
        if not d.exists():
            print(f'[!] {name} introuvable, ignoré'); continue
        names = load_names(d); class_names[name] = names
        print(f'{name} : listage des fichiers...')
        imgs = sorted(p for p in d.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXT)
        print(f'{name} : {len(imgs)} images, lecture parallèle ({N_THREADS} threads)')
        expected = set()
        with ThreadPoolExecutor(N_THREADS) as ex:
            for rec, lb, lbd, lp in tqdm(ex.map(process, [(name, d, names, p) for p in imgs]), total=len(imgs), desc=name):
                records.append(rec); boxes += lb; bad_lines += lbd; expected.add(lp)
        for t in d.rglob('*.txt'):
            if t.name.lower() in ('classes.txt', 'readme.txt'): continue
            if any(p.lower() == 'labels' for p in t.relative_to(d).parts) and t not in expected:
                orphan_labels.append(dict(dataset=name, file=str(t.relative_to(d))))
    pickle.dump((records, boxes, bad_lines, orphan_labels, class_names), open(CACHE, 'wb'))
    print('Cache sauvegardé :', CACHE)

df = pd.DataFrame(records)
bx = pd.DataFrame(boxes)
print(f'{len(df)} images scannées, {len(bx)} boîtes valides, {len(bad_lines)} lignes invalides, {len(orphan_labels)} labels orphelins')

REPORT['class_names'] = class_names
REPORT['images_per_dataset_split'] = df.groupby(['dataset', 'split']).size().rename('n').reset_index().to_dict('records') if len(df) else []
REPORT['label_status'] = df.groupby(['dataset', 'split', 'status']).size().rename('n').reset_index().to_dict('records') if len(df) else []
REPORT['corrupt_images'] = df[df.corrupt][['dataset', 'path']].head(30).to_dict('records') if len(df) else []
REPORT['n_corrupt'] = int(df.corrupt.sum()) if len(df) else 0
REPORT['invalid_label_lines'] = dict(total=len(bad_lines), examples=bad_lines[:20])
REPORT['orphan_labels'] = dict(total=len(orphan_labels), examples=orphan_labels[:20])

display(df.groupby(['dataset', 'split', 'status']).size().unstack(fill_value=0))

In [ ]:
# 6) Classes, tailles des boîtes, résolutions
if len(bx):
    cd = bx.groupby(['dataset', 'split', 'cls_name']).size().unstack('split', fill_value=0)
    cd['total'] = cd.sum(axis=1)
    display(cd)
    REPORT['class_distribution'] = cd.reset_index().to_dict('records')

    bx['micro'] = bx.area < 0.01   # boîte < 1 % de l'image
    sz = bx.groupby(['dataset', 'cls_name']).agg(n=('area', 'size'), area_median=('area', 'median'),
                                                  wpx_median=('wpx', 'median'), hpx_median=('hpx', 'median'),
                                                  pct_micro=('micro', 'mean')).round(4)
    display(sz)
    REPORT['bbox_stats_by_class'] = sz.reset_index().to_dict('records')

    # images avec plusieurs boîtes
    nb = df[df.status == 'labeled'].groupby('dataset').n_boxes.describe().round(2)
    display(nb)

res = df.dropna(subset=['w']).assign(res=lambda x: x.w.astype(int).astype(str) + 'x' + x.h.astype(int).astype(str))
top_res = res.groupby('dataset').res.value_counts().groupby(level=0).head(5)
print(top_res)
REPORT['top_resolutions'] = {k: v for k, v in top_res.groupby(level=0).apply(lambda s: s.droplevel(0).to_dict()).items()}

# Candidats "images normales" pour PatchCore
normal = df[df.status.isin(['empty_label', 'missing_label'])].groupby(['dataset', 'split', 'status']).size()
print('\nCandidats images normales (sans boîte) :'); print(normal)
REPORT['normal_candidates'] = normal.rename('n').reset_index().to_dict('records')

In [ ]:
# 7) Doublons exacts (md5), quasi-doublons (dhash) et fuite de données entre partitions/datasets
ok = df[~df.corrupt & df.md5.notna()].copy()

# --- exacts
exact = []
for md5, g in ok[ok.duplicated('md5', keep=False)].groupby('md5'):
    kind = ('cross_dataset' if g.dataset.nunique() > 1 else
            'cross_split' if g.split.nunique() > 1 else 'same_split')
    exact.append(dict(md5=md5, n=len(g), kind=kind, files=[f'{a}/{b}' for a, b in zip(g.dataset, g.path)][:6]))
ex_counts = collections.Counter(e['kind'] for e in exact)
print('Doublons exacts (groupes) :', dict(ex_counts))
REPORT['exact_duplicates'] = dict(groups_by_kind=dict(ex_counts), examples=[e for e in exact if e['kind'] != 'same_split'][:15])

# --- quasi-doublons
uniq = ok.drop_duplicates('md5').reset_index(drop=True)
near = dict(skipped=False)
if len(uniq) > MAX_IMAGES_NEAR_DUP:
    near['skipped'] = True; print('Quasi-doublons sautés (trop d images).')
else:
    arr = np.array([int(v) for v in uniq.dhash], dtype=np.uint64)
    tab = np.array([bin(i).count('1') for i in range(65536)], dtype=np.uint8)
    pairs, capped = [], False
    for i in range(len(arr) - 1):
        x = arr[i+1:] ^ arr[i]
        dist = tab[x.view(np.uint16).reshape(-1, 4)].sum(axis=1)
        for j in np.where(dist <= NEAR_DUP_MAX_HAMMING)[0]:
            pairs.append((i, i + 1 + int(j), int(dist[j])))
        if len(pairs) > 200000: capped = True; break
    kinds = collections.Counter()
    cross_examples = []
    for i, j, dd in pairs:
        a, b = uniq.iloc[i], uniq.iloc[j]
        k = ('cross_dataset' if a.dataset != b.dataset else
             'cross_split' if a.split != b.split else 'same_split')
        kinds[k] += 1
        if k != 'same_split' and len(cross_examples) < 30:
            cross_examples.append(dict(kind=k, dist=dd, a=f'{a.dataset}/{a.path}', b=f'{b.dataset}/{b.path}'))
    print('Quasi-doublons (paires) :', dict(kinds), '(tronqué)' if capped else '')
    near.update(pairs_by_kind=dict(kinds), capped=capped, examples=cross_examples[:15])
    print('NB : sur des textures très uniformes le dhash peut produire de faux quasi-doublons. À vérifier visuellement.')
REPORT['near_duplicates'] = near

# aperçu visuel de quelques paires cross-split / cross-dataset
import matplotlib.pyplot as plt
ex = near.get('examples', [])[:5]
if ex:
    fig, axes = plt.subplots(len(ex), 2, figsize=(7, 3 * len(ex)))
    axes = np.atleast_2d(axes)
    for r, e in enumerate(ex):
        for c, key in enumerate(['a', 'b']):
            try:
                axes[r, c].imshow(Image.open(ROOT / e[key])); axes[r, c].set_title(f"{e[key][:38]}\n{e['kind']} d={e['dist']}", fontsize=7)
            except Exception: pass
            axes[r, c].axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# 8) Métadonnées : dataset_metadata.xlsx / csv (utile pour éviter la fuite par pièce de tissu)
meta_files = []
for name in YOLO_DATASETS:
    d = ROOT / name
    if d.exists():
        meta_files += [p for p in d.rglob('*') if p.suffix.lower() in ('.xlsx', '.xls', '.csv') and 'audit' not in p.name.lower()]
REPORT['metadata_files'] = []
for f in meta_files:
    print('=' * 70); print(f.relative_to(ROOT))
    try:
        sheets = pd.read_excel(f, sheet_name=None) if f.suffix.lower() != '.csv' else {'csv': pd.read_csv(f)}
    except Exception as e:
        print('lecture impossible :', e); continue
    for sn, m in sheets.items():
        print(f'-- feuille "{sn}" : {m.shape[0]} lignes x {m.shape[1]} colonnes')
        print('colonnes :', list(m.columns))
        display(m.head(10))
        low = {c: m[c].value_counts(dropna=False).head(15).to_dict() for c in m.columns if m[c].nunique(dropna=False) <= 30}
        for c, v in low.items(): print(f'  {c}: {v}')
        REPORT['metadata_files'].append(dict(file=str(f.relative_to(ROOT)), sheet=sn, rows=int(m.shape[0]),
                                             columns=[str(c) for c in m.columns],
                                             head=m.head(5).astype(str).to_dict('records'),
                                             low_cardinality={str(c): {str(k): int(x) for k, x in v.items()} for c, v in low.items()}))

In [ ]:
# 9) DTD (Describable Textures Dataset) : catégories, splits officiels, intégrité
img_dir, lab_dir, imdb_dir = DTD_DIR / 'images', DTD_DIR / 'labels', DTD_DIR / 'imdb'
dtd = {}
if img_dir.exists():
    cats = sorted(d for d in img_dir.iterdir() if d.is_dir())
    counts = {c.name: sum(1 for f in c.iterdir() if f.suffix.lower() in IMG_EXT) for c in cats}
    dtd['n_categories'] = len(counts); dtd['n_images'] = sum(counts.values())
    dtd['min_max_per_category'] = [min(counts.values(), default=0), max(counts.values(), default=0)]
    print(f"DTD : {dtd['n_categories']} catégories, {dtd['n_images']} images")
    sp = {}
    for k in range(1, 11):
        try:
            s = {t: set(l.strip() for l in open(lab_dir / f'{t}{k}.txt') if l.strip()) for t in ('train', 'val', 'test')}
        except Exception:
            continue
        overlap = sum(len(s[a] & s[b]) for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')])
        sp[k] = dict(train=len(s['train']), val=len(s['val']), test=len(s['test']), overlap=overlap)
    dtd['official_splits'] = sp
    print('Splits officiels :', sp)
    try:
        s1 = set(l.strip() for t in ('train', 'val', 'test') for l in open(lab_dir / f'{t}1.txt') if l.strip())
        dtd['split1_missing_files'] = sum(1 for r in s1 if not (img_dir / r).exists())
        print('Fichiers du split 1 absents du disque :', dtd['split1_missing_files'])
    except Exception as e:
        print('split 1 non vérifié :', e)
else:
    print('DTD images/ introuvable dans', DTD_DIR)
dtd['labels_dir'] = sorted(p.name for p in lab_dir.iterdir())[:15] if lab_dir.exists() else None
dtd['imdb_dir'] = sorted(p.name for p in imdb_dir.iterdir())[:15] if imdb_dir.exists() else None
REPORT['dtd'] = dtd

In [ ]:
# 10) Graphiques
import matplotlib.pyplot as plt
if len(bx):
    for name in bx.dataset.unique():
        t = bx[bx.dataset == name].groupby(['cls_name', 'split']).size().unstack(fill_value=0)
        t.plot(kind='bar', stacked=True, figsize=(9, 4), title=f'{name} : boîtes par classe et partition')
        plt.tight_layout(); plt.savefig(OUT / f'classes_{name}.png', dpi=120); plt.show()
    plt.figure(figsize=(7, 4))
    for name in bx.dataset.unique():
        plt.hist(bx[bx.dataset == name].area, bins=np.logspace(-5, 0, 40), alpha=.5, label=name)
    plt.xscale('log'); plt.xlabel('aire normalisée de la boîte'); plt.legend(); plt.title('Taille des défauts (micro-défauts à gauche)')
    plt.tight_layout(); plt.savefig(OUT / 'bbox_area_hist.png', dpi=120); plt.show()

In [ ]:
# 11) Rapport final : COPIE-COLLE LA SORTIE DE CETTE CELLULE DANS LE CHAT (ou uploade audit_report.json)
def default(o):
    if hasattr(o, 'item'): return o.item()
    return str(o)
(OUT / 'audit_report.json').write_text(json.dumps(REPORT, indent=1, ensure_ascii=False, default=default), encoding='utf-8')
txt = json.dumps(REPORT, indent=1, ensure_ascii=False, default=default)
(OUT / 'audit_summary.txt').write_text(txt, encoding='utf-8')
print('Sauvegardé dans :', OUT)
print('=' * 70)
print(txt[:14000])
if len(txt) > 14000: print('\n[... tronqué : uploade audit_report.json pour le détail ...]')
try:
    from google.colab import files; files.download(str(OUT / 'audit_report.json'))
except Exception: pass